In [ ]:
import numpy as np
import torch 
import torch.nn as nn 
import scipy.io 
import torch
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


## Pytorch GPU Support in Apple Silicon 
    MPS Backend 
mps device enables high-performance training on GPU for MacOS devices with Metal programming framework. It introduces a new device to map Machine Learning computational graphs and primitives on highly efficient Metal Performance Shaders Graph framework and tuned kernels provided by Metal Performance Shaders framework respectively.

In [ ]:
# Check that MPS is available
if not torch.backends.mps.is_available():
    if not torch.backends.mps.is_built():
        print("MPS not available because the current PyTorch install was not "
              "built with MPS enabled.")
    else:
        print("MPS not available because the current MacOS version is not 12.3+ "
              "and/or you do not have an MPS-enabled device on this machine.")

else:
    mps_device = torch.device("mps:0")

## Load Data

In [ ]:
data = scipy.io.loadmat('/Users/ramtarun/Desktop/Cambridge/Indirect-Noise-in-Nozzles/Data/Data_PINN_ff0.01He0N551_subsonic.mat')

## Unpacking the Data
     Unpacking the labeled data from mat file, into pytorch tensors.

In [ ]:
U = torch.zeros((551,4), device=mps_device)
U[:,0] = torch.from_numpy(data['eta'])
U[:,1] = torch.from_numpy(data['pi_p'])
U[:,2] = torch.from_numpy(data['pi_m'])
U[:,3] = torch.from_numpy(data['sig'])

eta = U[:, 0]
pi_p = U[:, 1]
pi_m = U[:, 2]
sig = U[:, 3]

N1 = eta.shape[0]

D = torch.from_numpy(data['D'])
M = torch.from_numpy(data['M'])
u = torch.from_numpy(data['ubar'])
dMdx = torch.from_numpy(data['dMdx'])
dudx = torch.from_numpy(data['dudx'])
alpha = torch.from_numpy(data['alp'])
eta1 = torch.from_numpy(data['eta'])    

#print(N1)

meanflow = torch.zeros((N1,7), device=mps_device)
meanflow[:,0] = D
meanflow[:,1] = M
meanflow[:,2] = u
meanflow[:,3] = dMdx
meanflow[:,4] = dudx
meanflow[:,5] = alpha
meanflow[:,6] = eta1

#print(meanflow)
#meanflow.get_device()

## Data Preprocessing
    Converting the pytorch tensors into input dataset for Neural Networks, and splliting the dataset into Train and Test datase.

In [ ]:
he = torch.tensor([0.01]).to(mps_device)
HE = torch.tile(he, (1,N1)) # N x T
He = HE.flatten()[:,None]
ETA = eta.flatten()[:, None]
UU = U[:, 1:4]

#ETA_train.requires_grad = True
He_train, He_test = train_test_split(He, train_size = 0.9, shuffle = True)
Eta_train, Eta_test, UU_train, UU_test = train_test_split(ETA, UU, train_size=0.9, shuffle=False)
#print(UU)

meanflow_train, meanflow_test = train_test_split(meanflow, train_size=0.9, shuffle=False)



## Data Visualization 
    Plotting the Training data set

In [ ]:
plt.rcParams["figure.figsize"] = (15,5)
plt.rcParams["font.size"]  = 20
plt.plot(Eta_train.detach().cpu(), UU_train.detach().cpu())

In [ ]:
Eta_train.requires_grad = True
Eta_test.requires_grad = True
UU_train.requires_grad = True
UU_test.requires_grad = True

In [ ]:
def RHS_f(y, baseflow, f):
    
    pi_p, pi_m, sig = torch.unbind(y,axis=1)

    D = baseflow[:,0]
    M = baseflow[:,1]
    u = baseflow[:,2]
    dMdx = baseflow[:,3]
    dudx = baseflow[:,4]
    alpha = baseflow[:,5]
    eta1 = baseflow[:,6]    
    
    He = 0#a constant input

    gamma = 1.4#constant
    
    Msq = torch.square(M)

    Lambda = 1 + Msq * (gamma-1)/2
    zeta = f*gamma*Msq - 2*torch.tan(alpha)
    C1= ((gamma - 1)*(1-Msq)*f)/(2*Lambda*zeta)
    Ca = -C1*M*u*dMdx*(2-(2*Msq/(1-Msq)) - (2*gamma*Msq*(-2*f*Lambda - (gamma-1)/gamma *zeta)/(2*Lambda*zeta)))
    Ff = -(dudx + (4*f*u/(2*D)))
    
    denom = (M**2*u - u + C1*Msq *u + C1*Msq*Msq*gamma*u) #M**4
    vrh_p = M*(2*(1-M) + C1*M*(M-2+M*gamma*(1-2*M)))/(2*denom)
    vrh_m = -M*(2*(1+M) + C1*M*(M+2+M*gamma*(1+2*M)))/(2*denom)
    vkp_p = Msq*C1*(2+M*(gamma-1))/(2*denom)
    vkp_m = Msq*C1*(2-M*(gamma-1))/(2*denom)
    vth_p = C1*M*(M*(1+gamma) + M**2*(1-gamma) - 2)/denom
    vth_m = C1*M*(M*(1+gamma) - M**2*(1-gamma) + 2)/denom
    vsig = -(C1*Msq*(1+gamma*Msq) + Msq - 1)/denom
    calM = dMdx/(2*M)
    kp_p = (gamma - 1) + (2/M)
    kp_m = (gamma - 1) - (2/M)
    Gm_p = M*(Ca*(M + 1) + Ff*M*(C1*gamma*M*Msq + M + (1 - C1*Msq)))/(2*denom)
    Gm_m = M*(Ca*(M - 1) - Ff*M*(C1*gamma*M*Msq + M - (1 - C1*Msq)))/(2*denom)
    Ups = (Ca*(Msq - 1) - C1*Ff*Msq*Msq *(1+gamma))/denom
    
#     eq1 = - (2*np.pi*1j*He*vrh_p + Gm_m*kp_m + calM)*pi_p + (2*np.pi*1j*He*vkp_p + Gm_m*kp_p + calM)*pi_m - Gm_m*sig
#     eq2 = - (2*np.pi*1j*He*vkp_m + Gm_p*kp_m - calM)*pi_p - (2*np.pi*1j*He*vrh_m + Gm_p*kp_p + calM)*pi_m - Gm_m*sig
#     eq3 = - (2*np.pi*1j*He*vth_p + Ups*kp_m)*pi_p - (2*np.pi*1j*He*vth_m + Ups*kp_p)*pi_m - (2*np.pi*1j*He*vsig + Ups)*sig
#     eq1 = - tf.multiply((Gm_m*kp_m + calM),pi_p) + tf.multiply(( Gm_m*kp_p + calM),pi_m) - tf.multiply(Gm_m,sig)

    eq1 =  (Gm_m*kp_m + calM)*pi_p + ( Gm_m*kp_p - calM)*pi_m + Gm_m*sig
    eq2 =  ( Gm_p*kp_m - calM)*pi_p + ( Gm_p*kp_p + calM)*pi_m + Gm_p*sig
    eq3 =  ( Ups*kp_m)*pi_p + ( Ups*kp_p)*pi_m + (Ups)*sig
    
    return torch.stack([-eq1, -eq2, -eq3],axis=1)


#RHS_f(UU_train, meanflow_train, f=0.01)

In [ ]:
layers = np.array([2, 4, 4, 4, 3]) # hidden layers
Ntrain = Eta_train.shape[0]
Ntrain

In [ ]:
# Neural Network for Predicting the pi+, pi-, and si.

class NN(nn.Module):
    def __init__(self, X, layers):
        super().__init__()

        'activation function'
        self.activation = nn.Tanh()
    
        'Initialize neural network as a list using nn.Modulelist'  
        self.linears = nn.ModuleList([nn.Linear(layers[i], layers[i+1]) for i in range(len(layers)-1)])
    
        'Xavier Normal Initialization'
        for i in range(len(layers)-1):
            nn.init.xavier_normal_(self.linears[i].weight.data, gain=1.0)
            # set biases to zero
            nn.init.zeros_(self.linears[i].bias.data)

        self.X = X

    def forward(self, X):
        if torch.is_tensor(X) != True:         
            X = torch.from_numpy(X) 
        a = X.type(torch.float32)
        for i in range(len(layers) - 2):
            z = self.linears[i](a)
            a = self.activation(z)
        a = self.linears[-1](a)
        return a

In [ ]:
# Physics informed Neural Network 
f = torch.empty(1).to(device=mps_device)

class PINN(nn.Module):
    def __init__(self, X, uu, meanflow, layers):
        super().__init__()
        self.loss_function = nn.MSELoss()

        'Initialize our new parameters i.e.f (Inverse problem)' 
        self.f = torch.tensor([f], requires_grad=True).type(torch.float32)
        nn.init.zeros_(self.f)
        'Register f to optimize'
        self.f = nn.Parameter(self.f)
        'Initialize iterator'
        self.iter = 0
        'Call our DNN'
        self.dnn = NN(X, layers)
        'Register our new parameter'
        self.dnn.register_parameter('f', self.f)  
        
        self.meanflow = meanflow
        self.UU = uu
        self.X = X

        
    def loss_data(self, X):
        loss_u = self.loss_function(self.dnn(X), self.UU)
        return loss_u
    
    def loss_residual(self, X):
        f = self.f
        g = torch.clone(X)
        R = self.dnn(g)
        dr_dn = torch.autograd.grad(R, self.X, torch.ones_like(R), retain_graph=True, create_graph=True)[0]
        GR = RHS_f(R, self.meanflow, f)
        pde = dr_dn[:, 1:] + GR
        return torch.mean(pde**2)

    
    def Loss(self, X):
        loss_data = self.loss_data(self.X)
        loss_residual = self.loss_residual(self.X)
        return loss_residual + loss_data
    

    'callable for optimizer'                                       
    def closure(self):
        
        optimizer.zero_grad()
        
        loss = self.Loss(self.X)
        
        loss.backward()
                
        self.iter += 1
        
        if self.iter % 50 == 0:
            print('f_real = [0.01], f_PINN = [%.5f], Loss = [%.5f] ' %(self.f.item(), loss))
            
        return loss


In [ ]:
#torch.cuda.manual_seed(22)
learning_rate=1.0e-3
N_train = torch.cat([He_train, Eta_train],1)
N_test = torch.cat([He_test, Eta_test], 1)
PINN_model = PINN(N_train, UU_train, meanflow_train, layers)
PINN_model.to(mps_device)
params = list(PINN_model.parameters())
#optimizer = torch.optim.LBFGS(params, learning_rate, 
                              #max_iter = epochs, 
                              #max_eval = None, 
                              #tolerance_grad = 1e-11, 
                              #tolerance_change = 1e-11, 
                              #history_size = 100, 
                              #line_search_fn = 'strong_wolfe')
optimizer = torch.optim.Adam(params, lr = learning_rate, amsgrad = False, weight_decay=1e-6)

#PINN_model.loss_residual(N_train)


In [ ]:
epochs=250

In [ ]:
PINN_model.train()
epoch_loss = {}
for i in range(epochs):
    if i==0:
      print("Training Loss")
    loss = PINN_model.Loss(N_train)# use mean squared error
    optimizer.zero_grad()
    loss.backward()
    epoch_loss[i] = loss.item()
    optimizer.step(PINN_model.closure)

R_pred = PINN_model.dnn(N_train)

In [ ]:
plt.rcParams["figure.figsize"] = (20, 10)
plt.rcParams["font.size"]  = 20
plt.subplot(2,1,1)
plt.plot(epoch_loss.keys(), epoch_loss.values())
plt.title('MSE convergence')
plt.xlabel('Epochs')
plt.subplot(2,1,2)
plt.plot(Eta_train.detach().cpu(), UU_train.detach().cpu(), 'black')
plt.plot(Eta_train.detach().cpu(),R_pred.detach().cpu(), 'r--')